CUDA Testing to build model on Graphic Card

In [1]:
import tensorflow as tf
import numpy as np
import torch

print(f"Wersja TensorFlow: {tf.__version__}")
print(f"Wersja NumPy: {np.__version__}")
print("---")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"SUKCES! Znaleziono GPU: {gpus}")
else:
    print("BŁĄD: Nadal nie widzę GPU. Sprawdź czy foldery bin z CUDA i cuDNN są w zmiennych środowiskowych PATH.")


print(torch.cuda.is_available())

TypeError: Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

Confussion Matrix for YOLOv8

In [2]:
import cv2
import os
from ultralytics import YOLO

if __name__ == '__main__':
    model_path = r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\runs\detect\depthai_model\yolo_rocks_btr_noise\weights\best.pt"
    data_yaml_path = r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\data.yaml"

    model = YOLO(model_path)

    metrics = model.val(data=data_yaml_path)

    print(f"Wyniki zapisano w katalogu: {metrics.save_dir}")

    matrix_path = os.path.join(metrics.save_dir, "confusion_matrix.png")

    if os.path.exists(matrix_path):
        img = cv2.imread(matrix_path)

        img_resized = cv2.resize(img, (1024, 768))

        cv2.imshow("Macierz Bledow", img_resized)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    else:
        print("Nie znaleziono pliku z macierza bledow.")

Ultralytics 8.4.52  Python-3.10.11 torch-2.7.1+cu118 CUDA:0 (GeForce GTX 1650 Ti, 4096MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1800.6810.7 MB/s, size: 5781.6 KB)
val: Scanning D:\Studia\cybAIR\Ambition\rocks_detection\Detection\valid\labels.cache... 115 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 115/115  0.0s
val: D:\Studia\cybAIR\Ambition\rocks_detection\Detection\valid\images\20260601_174934_jpg.rf.LzfJioODWeDyZ4C3ZEiF.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.8it/s 4.5s0.4s
                   all        115       2227       0.82      0.689      0.766      0.502
Speed: 3.1ms preprocess, 11.1ms inference, 0.0ms loss, 13.3ms postprocess per image
Results saved to D:\Studia\cybAIR\Ambition\rocks_detection\Detection\runs\detect\val-12
Wyniki zapisano w katalogu: D:\Studia\cybAIR\Am

Convert YOLO to .blob

In [ ]:
import os
from ultralytics import YOLO
import blobconverter

model = YOLO(r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\runs\detect\depthai_model\yolo_rocks-5\weights\best.pt")

onnx_model_path = model.export(format="onnx", imgsz=416, opset=11, nms=False)
print(f"Model wyeksportowany do ONNX: {onnx_model_path}")

blob_path = blobconverter.from_onnx(
    model=onnx_model_path,
    data_type="FP16",
    shaves=6,
    version="2022.1",
    output_dir="depthai_model"
)

print(f"Sukces! Plik skompilowany do OAK-D Lite: {blob_path}")

Confussion Matrix for .blob

In [ ]:
#nothing here

Convert PNG Labels to TXT

In [2]:
import os
import cv2

def convert_masks_to_yolo_seg(masks_dir, labels_output_dir, class_id=0):
    os.makedirs(labels_output_dir, exist_ok=True)

    for mask_name in os.listdir(masks_dir):
        if mask_name.lower().endswith('.png'):
            mask_path = os.path.join(masks_dir, mask_name)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            if mask is None:
                continue

            h, w = mask.shape
            _, thresh = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            txt_lines = []
            for contour in contours:
                if cv2.contourArea(contour) < 10:
                    continue

                polygon_coords = []
                for point in contour:
                    pt_x, pt_y = point[0]
                    norm_x = pt_x / w
                    norm_y = pt_y / h
                    polygon_coords.append(f"{norm_x:.6f} {norm_y:.6f}")

                if polygon_coords:
                    coords_str = " ".join(polygon_coords)
                    txt_lines.append(f"{class_id} {coords_str}")

            if txt_lines:
                txt_name = os.path.splitext(mask_name)[0] + ".txt"
                txt_path = os.path.join(labels_output_dir, txt_name)
                with open(txt_path, "w") as f:
                    f.write("\n".join(txt_lines))

if __name__ == "__main__":
    base_dir = r"D:\Studia\cybAIR\Ambition\rocks_detection\Detection\notTested"

    splits = ["train", "val", "test"]

    for split in splits:
        masks_folder = os.path.join(base_dir, split, "images")
        labels_folder = os.path.join(base_dir, split, "labels")

        if os.path.exists(masks_folder):
            print(f"Przetwarzanie masek dla zestawu: {split}...")
            convert_masks_to_yolo_seg(masks_folder, labels_folder, class_id=0)

    print("Wszystkie maski zostały przekonwertowane do formatu YOLO-Segmentation!")

Przetwarzanie masek dla zestawu: train...
Przetwarzanie masek dla zestawu: val...
Przetwarzanie masek dla zestawu: test...
Wszystkie maski zostały przekonwertowane do formatu YOLO-Segmentation!


Converting our area detection of object to rectangle

In [1]:
import os

def convert_polygon_to_bbox(labels_dir):
    if not os.path.exists(labels_dir):
        return

    for filename in os.listdir(labels_dir):
        if not filename.endswith(".txt"):
            continue

        filepath = os.path.join(labels_dir, filename)

        with open(filepath, "r") as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) < 5:
                new_lines.append(line.strip())
                continue

            class_id = parts[0]

            coords = [float(x) for x in parts[1:]]
            xs = coords[0::2]
            ys = coords[1::2]

            if not xs or not ys:
                continue

            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)

            x_center = (x_min + x_max) / 2.0
            y_center = (y_min + y_max) / 2.0
            width = x_max - x_min
            height = y_max - y_min

            new_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        with open(filepath, "w") as f:
            f.write("\n".join(new_lines))

if __name__ == "__main__":
    base_dir = "."

    splits = ["train", "valid", "test"]

    for split in splits:
        labels_folder = os.path.join(base_dir, split, "labels")
        if os.path.exists(labels_folder):
            convert_polygon_to_bbox(labels_folder)

Increase data with noise

In [1]:
import os
import cv2
import shutil
import albumentations as A

IMG_DIR = './train/images'
LBL_DIR = './train/labels'
OUT_IMG_DIR = './trainNoise/images'
OUT_LBL_DIR = './trainNoise/labels'

os.makedirs(OUT_IMG_DIR, exist_ok=True)
os.makedirs(OUT_LBL_DIR, exist_ok=True)

transform = A.Compose([
    A.GaussNoise(std_range=(0.1, 0.25), p=0.7),
    A.MotionBlur(blur_limit=7, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.PixelDropout(dropout_prob=0.01, p=0.3)
])

for img_name in os.listdir(IMG_DIR):
    if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    img_path = os.path.join(IMG_DIR, img_name)
    image = cv2.imread(img_path)

    if image is None:
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    augmented = transform(image=image)
    aug_image = augmented['image']

    aug_image = cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR)

    base_name = os.path.splitext(img_name)[0]
    new_img_name = f"{base_name}_aug.jpg"
    new_lbl_name = f"{base_name}_aug.txt"

    new_img_path = os.path.join(OUT_IMG_DIR, new_img_name)
    cv2.imwrite(new_img_path, aug_image)

    lbl_path = os.path.join(LBL_DIR, f"{base_name}.txt")
    new_lbl_path = os.path.join(OUT_LBL_DIR, new_lbl_name)

    if os.path.exists(lbl_path):
        shutil.copy(lbl_path, new_lbl_path)

C:\Users\humus\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
